<a href="https://colab.research.google.com/github/KalitonOliveira001/KalitonOliveira001/blob/main/Projeto_Otimizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Comentários sobre o processo:
# - Leitura dos dados foi feita com parquet por ser mais leve e eficiente
# - Usamos tabelas temporárias para executar SQL com mais clareza
# - Utilizamos reparticionamento para melhorar a distribuição dos dados
# - O coalesce reduz o número de arquivos ao salvar
# - O select final filtra colunas desnecessárias, otimizando performance e armazenamento


In [78]:
from pyspark.sql import SparkSession

# Importando as bibliotecas necessárias
from pyspark.sql import SparkSession
# Criando uma SparkSession
spark = SparkSession.builder \
    .appName("Otimização de Join") \
    .getOrCreate()


In [79]:
#Uploaded dos arqivos
from google.colab import files
uploaded = files.upload()

Saving videos-preparados.snappy (1).parquet to videos-preparados.snappy (1).parquet


In [80]:
#Uploaded dos arqivos
from google.colab import files
uploaded = files.upload()

Saving videos-comments-tratados.snappy (1).parquet to videos-comments-tratados.snappy (1).parquet


In [94]:
   # 1 Ler o arquivo 'videos-preparados.snappy.parquet' no dataframe 'df_video'
   df_video = spark.read.parquet('videos-preparados.snappy.parquet')

In [95]:
   # 2 Ler o arquivo 'videos-comments-tratados.snappy.parquet' no dataframe 'df_comments'
   df_comments = spark.read.parquet('videos-comments-tratados.snappy.parquet')

In [96]:
# Passo 3: Criar tabelas temporárias para ambos os dataframes
df_video.createOrReplaceTempView("videos")
df_comments.createOrReplaceTempView("comments")

In [97]:
# Passo 4: Fazer um join das tabelas criadas anteriormente utilizando spark.sql
join_video_comments = spark.sql("""
    SELECT v.*, c.*
    FROM videos v
    JOIN comments c ON v.`Video ID` = c.`Video ID`
""")
# Exibindo o resultado do join
join_video_comments.show()

+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|               Title|   Video ID|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+--------------------+-----------+------------+-------+-----+--------+-------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23

In [101]:

# ETAPA 5: Repetir as etapas anteriores com **reparticionamento** e **coalesce**
# Reparticionamento para distribuir melhor os dados e otimizar a performance
df_video_reparticionado = df_video.repartition(4, "Video ID")
df_comments_reparticionado = df_comments.repartition(4, "Video ID")

# Criar tabelas temporárias novamente para o Spark SQL
df_video_reparticionado.createOrReplaceTempView("videos_rep")
df_comments_reparticionado.createOrReplaceTempView("comments_rep")

# Realizar o JOIN com os dados reparticionados
join_video_comments_reparticionado = spark.sql("""
    SELECT
        v.`Video ID`,
        v.Title,
        v.Likes,
        c.Comment,
        c.`Likes Comment`
    FROM videos_rep v
    JOIN comments_rep c ON v.`Video ID` = c.`Video ID`
""")

In [102]:

# ETAPA 6: Filtrar e otimizar o DataFrame após o JOIN
# Aqui selecionamos apenas as colunas necessárias para reduzir o tamanho dos dados
# e facilitar o armazenamento e processamento.
join_otimizado = join_video_comments_reparticionado.select(
    col("v.`Video ID`"),
    col("v.Title"),
    col("v.Likes"),
    col("c.Comment"),
    col("c.`Likes Comment`")
)

# Coalesce para reduzir o número de arquivos na escrita final
# (útil para exportações e evitar arquivos muito pequenos)
join_otimizado = join_otimizado.coalesce(1)

# Exibir algumas linhas para validação
join_otimizado.show(5)


+-----------+--------------------+-----+--------------------+-------------+
|   Video ID|               Title|Likes|             Comment|Likes Comment|
+-----------+--------------------+-----+--------------------+-------------+
|ErMwWXQxHp0|Best Back to Scho...|96513|Guys, a quick not...|        23964|
|ErMwWXQxHp0|Best Back to Scho...|96513|"this is hilariou...|          415|
|ErMwWXQxHp0|Best Back to Scho...|96513|Everyone has been...|           35|
|ErMwWXQxHp0|Best Back to Scho...|96513|"Shoutout to my O...|          233|
|ErMwWXQxHp0|Best Back to Scho...|96513|BEST BUY: We want...|         NULL|
+-----------+--------------------+-----+--------------------+-------------+
only showing top 5 rows



In [103]:
# ETAPA 7: Salvar o resultado final no formato Parquet
# O arquivo será salvo com nome 'join-videos-comments-otimizado'
join_otimizado.write.mode("overwrite").parquet("/content/join-videos-comments-otimizado")


In [60]:
#Encerrar  sessão
spark.stop()